In [1]:
import yaml
from app.datasets.loader import load_multiple_test_cases, load_test_cases
from app.datasets.validator import validate_dataset_schema
from app.client.rag_client import RAGClient
from app.tests.run_tests import run_tests
from app.tests.nodes.reformulate import send_reformulate_requests, save_reformualte_responses, reformulate_tests

In [2]:
file_list = [
  './app/data/raw/rapido.xlsx',
]

test_config = {
  'GENERAL_TESTS': True,
  'TIMINGS': {'test': True, 'report': False},
  'TOKENS': {'test': True, 'report': False},
  'FOUNDRYS': {'test': True, 'report': False},
  'TRIAGE': {'test': True, 'report': False},
  'ROUTER': {'test': True, 'report': False},
  'GROUNDING': {'test': True, 'report': False},
  'SAVE_RESULTS': False,
  'PATH': './app/data/processed/reports/report_RAPIDO',
  
  'REFORMULATE': {'test': True, 'report': False}
}   

if file_list: 
  df = load_multiple_test_cases(file_list)
  df = validate_dataset_schema(df)

with open('./app/config/config.yaml', 'r') as file:
  config_data = yaml.load(file, Loader= yaml.FullLoader) 
  
client = RAGClient(config_data)
test_timestamps = {}

In [3]:
# TEST GENERALES
if test_config.get('GENERAL_TESTS', False):
  responses = client.query_batch(df['user_input'],df['reference'])
  save_responses_in_json, response_file_path = client.save_api_responses(responses)
  test_timestamps['general_tests'] = str(response_file_path).replace('\\', '/')

Processing queries:  70%|███████   | 7/10 [01:20<00:38, 12.81s/it]

Max retries exceeded
Network Error: 'NoneType' object has no attribute 'get'


Processing queries:  80%|████████  | 8/10 [01:45<00:33, 16.62s/it]

Max retries exceeded
Network Error: 'NoneType' object has no attribute 'get'


Processing queries: 100%|██████████| 10/10 [02:00<00:00, 12.02s/it]


In [ ]:
import json

response_file_path = './app/data/processed/outcomes/outcome_20260412-210310.json'
with open(response_file_path, 'r', encoding='UTF-8') as f:
  responses = json.load(f)
test_timestamps['general_tests'] = 'outcome_20260412-210310.json'

In [5]:
if test_config.get('GENERAL_TESTS'):
  results, reports = run_tests(
    config = test_config, 
    data = responses, 
    df = df, 
    timestamp = test_timestamps
)

In [6]:
print(results)

{'timestamp': '20260412-210617', 'nodes': {'triage': {'positives': 8, 'total': 8, 'result': 100.0}, 'router': {'positives': 8, 'total': 8, 'result': 100.0}, 'grounding': {'positives': 8, 'total': 8, 'result': 100.0}}, 'timings': {'reformulate': {'prom': 0.991, 'p90': 1.316, 'p95': 1.372, 'quantity': 8}, 'triage': {'prom': 0.992, 'p90': 1.178, 'p95': 1.179, 'quantity': 8}, 'router': {'prom': 0.365, 'p90': 0.467, 'p95': 0.467, 'quantity': 8}, 'ag_call': {'prom': 3.909, 'p90': 6.242, 'p95': 6.269, 'quantity': 8}, 'personality': {'prom': 2.092, 'p90': 3.203, 'p95': 3.393, 'quantity': 8}, 'grounding': {'prom': 1.567, 'p90': 2.128, 'p95': 2.195, 'quantity': 8}, 'retriever': {'prom': 1.146, 'p90': 2.264, 'p95': 3.745, 'quantity': 8}, 'ret_embeddings': {'prom': 0.874, 'p90': 2.0, 'p95': 3.507, 'quantity': 8}, 'rag_answer': {'prom': 2.486, 'p90': 4.616, 'p95': 5.166, 'quantity': 8}, 'response_time': {'prom': 8.122, 'p90': 9.785, 'p95': 10.835, 'quantity': 8}}, 'tokens': {'in_ref': {'prom': 929.